# Transfer learning en PyTorch: notebook breve y didáctico

Este notebook muestra el patrón esencial de *transfer learning*:

1. cargar una red **preentrenada**,
2. **congelar** parte de sus capas,
3. **reemplazar** la cabeza final,
4. entrenar solo la nueva cabeza sobre una **tarea objetivo con pocos datos**,
5. opcionalmente, hacer un **fine-tuning** suave de las últimas capas.

La idea está alineada con tus transparencias: reutilizar conocimiento de una red ya entrenada, dejar parte de las capas como **no entrenables** y entrenar una parte pequeña para una tarea nueva.

## 0. Comentario didáctico

En las transparencias se presenta el escenario conceptual:

- una red para una **tarea fuente** ya entrenada,
- una **tarea objetivo** similar pero con pocos datos,
- las primeras capas se reutilizan,
- una parte de la red queda **non-trainable** y otra **trainable**.

En práctica con PyTorch, el patrón más común es:

- usar un modelo de `torchvision` preentrenado en ImageNet,
- congelar el *backbone*,
- sustituir la última capa (`fc` o clasificador),
- entrenar la nueva cabeza con el dataset objetivo,
- y, si interesa, descongelar las últimas capas para refinar.

In [ ]:
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

## 1. Dataset objetivo pequeño

Para que el ejemplo sea breve, usamos **CIFAR10** y nos quedamos solo con dos clases:

- `cat`
- `dog`

Además reducimos el tamaño del conjunto de entrenamiento para imitar el caso de **pocos datos**.

> Nota: este ejemplo necesita poder descargar CIFAR10 y, si se usan pesos preentrenados, también los pesos del modelo.

In [ ]:
# Transformaciones compatibles con ResNet18 preentrenada en ImageNet
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

# Dataset completo
train_full = datasets.CIFAR10(root="./data", train=True, download=True)
test_full  = datasets.CIFAR10(root="./data", train=False, download=True)

class_names = train_full.classes
print(class_names)

In [ ]:
# Nos quedamos solo con cat y dog
target_classes = {3: 0, 5: 1}  # 3=cat, 5=dog  -> remapeamos a 0/1

def filter_indices(dataset, allowed_classes):
    idx = []
    for i, (_, y) in enumerate(dataset):
        if y in allowed_classes:
            idx.append(i)
    return idx

train_idx_all = filter_indices(train_full, target_classes)
test_idx_all  = filter_indices(test_full, target_classes)

print("Train cat/dog total:", len(train_idx_all))
print("Test  cat/dog total:", len(test_idx_all))

In [ ]:
# Subconjuntos pequeños para imitar "small target dataset"
# Puedes aumentar estos tamaños si quieres más rendimiento.
n_train_small = 1000
n_val_small   = 200
n_test_small  = 400

train_idx_all = train_idx_all[: n_train_small + n_val_small]
test_idx_all  = test_idx_all[: n_test_small]

train_idx = train_idx_all[:n_train_small]
val_idx   = train_idx_all[n_train_small:n_train_small + n_val_small]
test_idx  = test_idx_all

# Wrapper para aplicar transformaciones y remapear etiquetas
class BinaryCIFAR10(torch.utils.data.Dataset):
    def __init__(self, base_dataset, indices, transform=None, label_map=None):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform
        self.label_map = label_map or {}

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base_dataset[self.indices[idx]]
        if self.transform is not None:
            x = self.transform(x)
        y = self.label_map[y]
        return x, y

train_ds = BinaryCIFAR10(train_full, train_idx, transform=preprocess, label_map=target_classes)
val_ds   = BinaryCIFAR10(train_full, val_idx,   transform=preprocess, label_map=target_classes)
test_ds  = BinaryCIFAR10(test_full,  test_idx,  transform=preprocess, label_map=target_classes)

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# Visualización rápida
inv_mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
inv_std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def show_batch(loader, n=6):
    x, y = next(iter(loader))
    fig, axes = plt.subplots(1, n, figsize=(14, 3))
    for i in range(n):
        img = x[i] * inv_std + inv_mean
        img = img.clamp(0, 1).permute(1, 2, 0)
        axes[i].imshow(img)
        axes[i].set_title("cat" if y[i].item() == 0 else "dog")
        axes[i].axis("off")
    plt.tight_layout()

show_batch(train_loader)

## 2. Modelo preentrenado y reemplazo de la cabeza final

Aquí está el corazón del *transfer learning*.

Usamos `resnet18` preentrenada y cambiamos su última capa para que la salida tenga tamaño 2:

- clase 0 = cat
- clase 1 = dog

In [ ]:
model = models.resnet18(weights=weights)

# Reemplazamos la capa final
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 2)

model = model.to(device)
print(model.fc)

## 3. Congelar el backbone

Primero congelamos **todas** las capas y luego reactivamos solo la cabeza final.

Esto implementa exactamente la idea de las transparencias:

- parte de la red = **non-trainable**
- parte de la red = **trainable**

In [ ]:
# Congelar todo
for param in model.parameters():
    param.requires_grad = False

# Descongelar solo la cabeza final
for param in model.fc.parameters():
    param.requires_grad = True

# Comprobación
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Parámetros totales:     {total_params:,}")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

## 4. Funciones de entrenamiento y evaluación

Son las mismas ideas que ya has visto en otros notebooks:

- `train_one_epoch`: forward, loss, backward, update
- `evaluate`: modo evaluación y sin gradientes

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        total += y.size(0)
        correct += (pred == y).sum().item()

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        running_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        total += y.size(0)
        correct += (pred == y).sum().item()

    return running_loss / total, correct / total

## 5. Fase 1: entrenar solo la cabeza nueva

En esta primera fase el *backbone* está congelado y solo aprende `model.fc`.

In [ ]:
num_epochs_head = 3

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
}

t0 = time.time()
for epoch in range(num_epochs_head):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(
        f"[Head] Época {epoch+1}/{num_epochs_head} | "
        f"train loss={train_loss:.4f}, acc={train_acc:.4f} | "
        f"val loss={val_loss:.4f}, acc={val_acc:.4f}"
    )
t1 = time.time()

print(f"Tiempo fase cabeza: {t1 - t0:.1f} s")

## 6. Fase 2 (opcional): fine-tuning suave

Una vez que la cabeza ya aprendió algo, a menudo conviene **descongelar las últimas capas** del backbone y seguir entrenando con una tasa de aprendizaje pequeña.

Aquí, por ejemplo, descongelamos `layer4` y `fc`.

In [ ]:
# Descongelar solo la última etapa convolucional y la cabeza
for name, param in model.named_parameters():
    if name.startswith("layer4") or name.startswith("fc"):
        param.requires_grad = True
    else:
        param.requires_grad = False

# Ver qué partes quedan entrenables
for name, param in model.named_parameters():
    if param.requires_grad:
        print("Trainable:", name)

In [ ]:
optimizer_ft = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

num_epochs_ft = 2

for epoch in range(num_epochs_ft):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer_ft, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(
        f"[Fine-tuning] Época {epoch+1}/{num_epochs_ft} | "
        f"train loss={train_loss:.4f}, acc={train_acc:.4f} | "
        f"val loss={val_loss:.4f}, acc={val_acc:.4f}"
    )

## 7. Evaluación final en test

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test loss = {test_loss:.4f}")
print(f"Test acc  = {test_acc:.4f}")

## 8. Curvas de aprendizaje

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], marker="o", label="train")
plt.plot(epochs, history["val_loss"], marker="o", label="val")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Pérdida")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_acc"], marker="o", label="train")
plt.plot(epochs, history["val_acc"], marker="o", label="val")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## 9. Comparación conceptual: entrenamiento directo vs transfer learning

Para conectar con las transparencias, aquí está la diferencia conceptual:

### Entrenamiento directo
- inicializas la red al azar,
- todos los pesos empiezan sin conocimiento previo,
- necesitas más datos y más entrenamiento.

### Transfer learning
- empiezas desde una red preentrenada,
- reutilizas representaciones ya útiles,
- congelas una parte y entrenas otra,
- suele funcionar mejor cuando la tarea objetivo tiene pocos datos.

En las transparencias, esta idea aparece como:

- **source model / source data** con muchos datos,
- **target model / target data** con pocos datos,
- transferencia de las primeras capas o de un bloque de capas.

## 10. Variante muy corta: plantilla mínima reutilizable

Si solo quieres recordar el patrón, esta es la plantilla esencial:

```python
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, num_classes)

for p in model.fc.parameters():
    p.requires_grad = True

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
```

Y después:

1. entrenas la cabeza,
2. opcionalmente descongelas `layer4`,
3. haces fine-tuning con `lr` más pequeña.

## 11. Ejercicio sugerido para alumnos

1. Repetir el notebook con:
   - `n_train_small = 200`
   - `n_train_small = 1000`
   - `n_train_small = 5000`

2. Comparar:
   - entrenamiento directo (`weights=None`)
   - transfer learning (`weights=DEFAULT`)

3. Medir:
   - accuracy final
   - velocidad de convergencia
   - estabilidad de validación

Así se ve muy bien cuándo el transfer learning ayuda más.